### TMDB Movies EDA

Dataset : https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies

##### Environment Setup & Data Loading

Importing libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns   

import plotly.express as px
import plotly.graph_objects as go   
from plotly.subplots import make_subplots 

# import plotly.io as pio
# pio.renderers.default='notebook_connected'

Read Data from CSV

In [ ]:
data = pd.read_csv('Dataset/TMDB Movies Dataset.csv', usecols=['id', 'title', 'vote_average', 'vote_count', 'status', 'revenue', 'release_date', 'runtime', 'budget', 'overview', 'genres', 'original_language', 'production_companies', 'production_countries', 'spoken_languages', 'keywords', 'homepage', 'imdb_id', 'popularity', 'tagline'])

print("Original Data Shape : ", data.shape)

Random Sampling

In [ ]:
# data = data.sample(n=300000, random_state=42)

# print("Sampling Data Shape : ", data.shape)

Display first 5 records from dataset

In [ ]:
display(data.head(5))

Display last 2 records from dataset

In [ ]:
display(data.tail(2))

##### Data Overview

Display total number of Rows and Columns from the dataset

In [ ]:
row, col = data.shape
print(f"Rows : {row}\nColumns : {col}")

Dataset Informations

In [ ]:
inf = data.info()
print(inf)

Dataset Column names

In [ ]:
col_names = data.columns    
print(col_names)

Descriptive Statistics of Dataset

In [ ]:
stat = data.describe(include='all')
display(stat)

Displaying columns along with their data types

In [ ]:
print(data.dtypes)

#### Observatrions from Data Types

- Converting release_date to datetime from object
- genres are stored as object but contains structured list data so convert it into list
- production_companies, production_countries, spoken_languages are object types but contains object data so required json like structured data

Required Data Type Change

In [ ]:
# Convert release_date to datetime

data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')

# Extract year and month from release date

data['release_year'] = data['release_date'].dt.year
data['release_month'] = data['release_date'].dt.month

# convert json to python object 

list_columns = [
    'genres',
    'production_companies',
    'production_countries',
    'spoken_languages',
    'keywords'
]

for col in list_columns:
    data[col] = data[col].apply(
        lambda x: x if isinstance(x, list) 
        else x.split(', ') if isinstance(x, str) 
        else []
    )

# Extract primary genre (Feature Engineering)

data['primary_genre'] = data['genres'].apply(
    lambda x: x[0] if len(x) > 0 else np.nan
)

# Convert Categorical columns to categorical type

categorical_columns = ['original_language', 'status', 'primary_genre']

for col in categorical_columns:
    data[col] = data[col].astype('category')

print(data.dtypes)

Selecting a Rows by Movie name label

In [ ]:
# row = data.loc[['Kizingo', 'Deadpool']]
# display(row)

Selecting Specific Rows and Columns

In [ ]:
# selection = data.loc[['Kizingo', 'Deadpool'],['id', 'release_date', 'genres']]
# display(selection)

Selecting a Rows and Columns by Position (iloc) : 

In [ ]:
# selection = data.iloc[[3,5,7],[0,4,14]]
# display(selection)

##### Data Cleaning & Preprocessing

In [ ]:
na = data.isnull().sum().sort_values(ascending=False)

display(na)

Missing Value Analysis

In [ ]:
missing = pd.DataFrame({
    'Missing Values': data.isnull().sum(),
    'Percentage (%)': (data.isnull().sum() / len(data)) * 100
}).sort_values(by='Percentage (%)', ascending=False)
print(missing)

missing_filtered = missing[missing['Missing Values'] > 0].reset_index()
missing_filtered.rename(columns={'index': 'Feature'}, inplace=True)

fig = px.bar(
    missing_filtered,
    x='Feature',
    y='Percentage (%)',
    title='Missing Value % per Feature',
    color_discrete_sequence=["#10d421"],
)


fig.show()
fig.write_image("missing_value_analysis.png")

Duplicate Records

In [ ]:
print(f"Duplicate rows (by id): {data.duplicated(subset='id').sum()}")
print(f"Unique movies (by id): {data['id'].nunique()}")

Drop Duplicate Data

In [ ]:
data.drop_duplicates(subset='id', inplace=True)
print(data.shape)

Drop Columns which is not used for analytical value (homepage, tagline) 

In [ ]:
# Homepage and tagline are optional marketing attributes and do not directly impact financial analysis so drop it.

data.drop(columns=['homepage', 'tagline'], inplace=True)
print("Columns after Drop : ",data.columns)

Keyword Data is using for NLP so fill empty field with empty list.

In [ ]:
data['keywords'].fillna("Unknown", inplace=True)
print('Keyword Columns Missing Data : ',data['keywords'].isna().sum())

Production Companies, Production Countries and Spoken Language data are filled with 'Unknown' instead of dropping rows and retain dataset size while preserving categorical consistency.

In [ ]:
data['production_companies'].fillna("Unknown", inplace=True)
data['production_countries'].fillna("Unknown", inplace=True)
data['spoken_languages'].fillna("Unknown", inplace=True)

Overview data is filled with 'No Overview Available' for NLP safety

In [ ]:
data['overview'].fillna("No Overview Available", inplace=True)
print('Overview Columns Missing Data : ',data['overview'].isna().sum())

In [ ]:
na = data.isnull().sum().sort_values(ascending=False)

display(na)

Replace 0 → NaN for financial / metric columns

In [ ]:
# These columns report 0 nulls but 0 values ARE the missing data

# if we keep 0 in this features it destroys average or ROI becomes wrong

zero_means_missing = ['budget', 'revenue', 'runtime']
for col in zero_means_missing:
    before = (data[col] == 0).sum()
    data[col] = data[col].replace(0, np.nan)
    print(f"{col}: replaced {before:,} zeros → NaN")


vote_zeros = (data['vote_average'] == 0.0).sum()
data['vote_average'] = data['vote_average'].replace(0.0, np.nan)

print(f"vote_average: replaced {vote_zeros:,} zero ratings → NaN")

print("\n── True missing after zero replacement ──")
print(data[zero_means_missing + ['vote_average']].isnull().sum())

Feature Engineering

In [ ]:
data['profit']        = data['revenue'] - data['budget']

data['roi']           = ((data['revenue'] - data['budget']) / data['budget']) * 100

data['is_profitable'] = data['profit'] > 0

print(data.columns)

##### Univariate Analysis

Distribution of Budget

In [ ]:
n_data = data['budget'].replace(0, np.nan).dropna()

# Create figure
plt.figure(figsize=(10, 6))

# Plot histogram
plt.hist(
    n_data,
    bins=40,
    color='#7b61ff',
    alpha=0.85,
    edgecolor='white'
)

# Titles and labels
plt.title('Distribution of Budget', fontsize=14, fontweight='bold')
plt.xlabel('Budget')
plt.ylabel('Movies')

# Dark theme styling
plt.gca().set_facecolor('#12121a')
plt.gcf().set_facecolor('#0a0a0f')
plt.gca().tick_params(colors='#e8e8f0')
plt.title('Distribution of Budget', color='#e8e8f0')
plt.xlabel('Budget', color='#e8e8f0')
plt.ylabel('Movies', color='#e8e8f0')

plt.tight_layout()
plt.show()

# Budget & revenue are:
# Extremely right-skewed
# Few blockbuster movies dominate

# Log transformation:

# Reduces skewness
# Makes distribution more normal
# Makes visualization readable

data['budget_log'] = np.log1p(data['budget'])


plt.figure(figsize=(10, 6))

plt.hist(
    data['budget_log'].dropna(),
    bins=40,
    edgecolor='black'
)

plt.title('Log Distribution of Budget')
plt.xlabel('Log(Budget)')
plt.ylabel('Number of Movies')

plt.grid(alpha=0.3)
plt.show()


Distribution of Revenue

In [ ]:
n_data = data['revenue'].replace(0, np.nan).dropna()

# Create figure
plt.figure(figsize=(10, 6))

# Plot histogram
plt.hist(
    n_data,
    bins=40,
    color='#7b61ff',
    alpha=0.85,
    edgecolor='white'
)

# Titles and labels
plt.title('Distribution of Revenue', fontsize=14, fontweight='bold')
plt.xlabel('Revenue')
plt.ylabel('Movies')

# Dark theme styling
plt.gca().set_facecolor('#12121a')
plt.gcf().set_facecolor('#0a0a0f')
plt.gca().tick_params(colors='#e8e8f0')
plt.title('Distribution of Revenue', color='#e8e8f0')
plt.xlabel('Revenue', color='#e8e8f0')
plt.ylabel('Movies', color='#e8e8f0')

plt.tight_layout()
plt.show()

# Budget & revenue are:
# Extremely right-skewed
# Few blockbuster movies dominate

# Log transformation:

# Reduces skewness
# Makes distribution more normal
# Makes visualization readable

data['revenue_log'] = np.log1p(data['revenue'])
plt.figure(figsize=(10, 6))

plt.hist(
    data['revenue_log'].dropna(),
    bins=40,
    edgecolor='black'
)

plt.title('Log Distribution of Revenue')
plt.xlabel('Log(Revenue)')
plt.ylabel('Number of Movies')

plt.grid(alpha=0.3)
plt.show()

Distribution of Vote Average

In [ ]:
new_data = data['vote_average'].replace(0, np.nan).dropna()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=new_data,
    nbinsx=40,
    name='Vote Average',
    marker_color='#7b61ff',
    opacity=0.85,
    hovertemplate='Rating: %{x}<br>Count: %{y}<extra></extra>'
))

fig.add_vline(
    x=new_data.mean(),
    line_dash='dash',
    line_color='#f5a623',
    annotation_text=f'Mean: {new_data.mean():.2f}',
    annotation_position='top right'
)

fig.add_vline(
    x=new_data.median(),
    line_dash='dot',
    line_color='#00d4b4',
    annotation_text=f'Median: {new_data.median():.2f}',
    annotation_position='top left'
)

fig.update_layout(
    title='Distribution of Vote Average',
    xaxis_title='Vote Average',
    yaxis_title='Count',
    height=500,
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#12121a',
    font_color='#e8e8f0',
    bargap=0.05
)

fig.show()
fig.write_image("vote_average_distribution.png")

The Vote Average Distribution is Right Skewed which indicating that most movies have high vote(6-10). The Mean vote is 6.15, which is slightly higher than Median vote 6.0 so ditribution is Right Skewed.

In [ ]:
# Calculate mean and median
mean_vote = data['vote_average'].mean()
median_vote = data['vote_average'].median()

print(f"Mean Vote Average: {mean_vote}")
print(f"Median Vote Average: {median_vote}")

##### Bivariate & Multivariate Analysis

Movies Status

In [ ]:
status_counts = data['status'].value_counts().reset_index()
status_counts.columns = ['status', 'count']
fig = px.pie(
    status_counts, names='status', values='count',
    title='Movie Status Breakdown',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.4                     
)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.show()
fig.write_image("movies_status_diagrams.png")

Movies Languages

In [ ]:
lang_df = data['original_language'].value_counts().head(15).reset_index()
lang_df.columns = ['language', 'count']

lang_df = lang_df.sort_values('count')

plt.figure(figsize=(10, 7))

bars = plt.barh(
    lang_df['language'],
    lang_df['count']
)

plt.title('Top 15 Original Languages')
plt.xlabel('Count')
plt.ylabel('Language')

for index, value in enumerate(lang_df['count']):
    plt.text(value, index, f' {value}', va='center')

plt.tight_layout()
plt.show()

Insights by Movie Languages : 

The dataset reflects strong representation of English-language films, highlighting Hollywood's global influence.

Normalize Popularity : 

Transform popularity data using Min-Max Scaling Normalization because if we scale data between 1-10 then comparision between popularity and vote_average becomes easy and accurate.

Because vote_average data already scaled between 1-10 and popularity data is unbounded.

In [ ]:
pop_min = data['popularity'].min()
pop_max = data['popularity'].max()

print("Popularity Before Scaled")
print(f"Min : {pop_min}, Max : {pop_max}")

data['popularity_scaled'] = 1 + 9 * (
    (data['popularity'] - pop_min) / (pop_max - pop_min)
)

pop_min = data['popularity_scaled'].min()
pop_max = data['popularity_scaled'].max()

print("Popularity After Scaled")
print(f"Min : {pop_min}, Max : {pop_max}")

# Comparison between polularity and vote_average

fig = px.scatter(
    data, x='popularity_scaled', y='vote_average',
    color='vote_count', color_continuous_scale='Plasma',
    title='Popularity vs Rating (coloured by vote count)',
    log_x=True
)
fig.show()
fig.write_image("popularity_vote_average.png")

Budget vs Profit Comparison

In [ ]:
fig = px.scatter(
    data, x='budget', y='profit',
    color='profit', color_continuous_scale='Plasma',
    title='Budget vs Profit (coloured by profit)',
    log_x=True
)
fig.show()
fig.write_image("budget_profit_comparison.png")

##### Text & Categorical Feature Analysis

Top Movie Genres

In [ ]:
# Genre Frequency

genres_exploded = data.explode('genres')
genre_counts = genres_exploded['genres'].value_counts().head(15)

genre_counts = genre_counts.sort_values()

plt.figure(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(genre_counts)))

bars = plt.barh(
    genre_counts.index,
    genre_counts.values,
    color=colors
)

plt.title('Top 15 Movie Genres')
plt.xlabel('Number of Movies')
plt.ylabel('Genre')

for i, value in enumerate(genre_counts.values):
    plt.text(value, i, f' {value}', va='center')

plt.tight_layout()
plt.show()


Insights by Movies Genre : 

High-frequency genres such as Drama and Documentary dominate the industry, indicating strong and consistent audience demand.

Top Keywords TreeMap

In [ ]:
# Top Keywords Treemap

keywords_explode = data.explode('keywords')
keywords_count = keywords_explode['keywords'].value_counts().head(20)

fig = px.treemap(
    names=keywords_count.index,
    parents=[""] * len(keywords_count),
    values=keywords_count.values,
    title='Top 20 Movie Keywords (Treemap)',
    color_continuous_scale='Teal'
)

fig.update_layout(title_x=0.5)
fig.show()
fig.write_image("keywords_treemap.png")

Top Production Companies 

In [ ]:
companies_exploded = data.explode('production_companies')
company_counts = companies_exploded['production_companies'].value_counts().head(15)

company_counts = company_counts.sort_values()

plt.figure(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(company_counts)))

bars = plt.barh(
    company_counts.index,
    company_counts.values,
    color=colors
)

plt.title('Top Production Companies')
plt.xlabel('Number of Movies')
plt.ylabel('Company')

for i, value in enumerate(company_counts.values):
    plt.text(value, i, f' {value}', va='center')

plt.tight_layout()
plt.show()

Top Companies by Average Revenue

In [ ]:
companies = data.explode('production_companies')

company_revenue = companies.groupby('production_companies')['revenue'].mean().sort_values(ascending=False).head(10)

company_revenue = company_revenue.sort_values()

company_revenue.plot(kind='barh')
plt.title('Top companies by Average Revenue')
plt.xlabel('Average Revenue (In Billion)')
plt.ylabel('Company')

for i, value in enumerate(company_revenue.values):
    plt.text(value, i, f' {value/1e9:.1f}B', va='center')

plt.show()

In [ ]:
data = data.reset_index(drop=True)

sns.pairplot(
    data[['budget_log','revenue_log','popularity_scaled','vote_average']],
    diag_kind='kde',
    plot_kws={'alpha':0.6},
)

plt.show()

Genre Impact on Popularity & Revenue

In [ ]:
genres_exploded = data.explode('genres')

# Popularity by Genre
genres_popularity = (genres_exploded.groupby('genres')['popularity'].mean().sort_values(ascending=False).head(15))

plt.figure(figsize=(12,6))
sns.barplot(
    x=genres_popularity.values,
    y=genres_popularity.index
)

for i, value in enumerate(genres_popularity.values):
    plt.text(value, i, f' {value:.1f}', va='center')



plt.title('Average Popularity by Genres')
plt.xlabel('Average Popularity')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()


Genre Revenue vs Genre Popularity

In [ ]:
genres_exploded = data.explode('genres')

genre_summary = (
    genres_exploded
    .groupby('genres')
    .agg({
        'revenue': 'mean',
        'popularity': 'mean',
        'id': 'count'
    })
    .reset_index()
)


genre_summary.columns = ['Genre', 'Avg_Revenue', 'Avg_Popularity', 'Movie_Count']

rev_median = genre_summary['Avg_Revenue'].median()
pop_median = genre_summary['Avg_Popularity'].median()


def classify(row):
    if row['Avg_Revenue'] >= rev_median and row['Avg_Popularity'] >= pop_median:
        return "High Revenue & High Popularity"
    elif row['Avg_Revenue'] >= rev_median and row['Avg_Popularity'] < pop_median:
        return "High Revenue & Low Popularity"
    elif row['Avg_Revenue'] < rev_median and row['Avg_Popularity'] >= pop_median:
        return "Low Revenue & High Popularity"
    else:
        return "Low Revenue & Low Popularity"

genre_summary['Category'] = genre_summary.apply(classify, axis=1)

fig = px.scatter(
    genre_summary,
    x='Avg_Popularity',
    y='Avg_Revenue',
    size='Movie_Count',
    color='Category',
    hover_name='Genre',
    title='Genre Classification: Revenue vs Popularity',
    size_max=60
)

fig.add_hline(y=rev_median, line_dash="dash")
fig.add_vline(x=pop_median, line_dash="dash")

fig.update_layout(
    xaxis_title="Average Popularity",
    yaxis_title="Average Revenue",
    height=650
)

fig.show()
fig.write_image("genre_revenue_popularity.png")

Genre by Average Revenue

In [ ]:
genre_revenue = genres_exploded.groupby('genres')['revenue'].mean().sort_values(ascending=False).head(15)

fig = px.bar(
    x=genre_revenue.index,
    y=genre_revenue.values,
    title='Average Revenue by Genre',
    labels={'x': 'Genre', 'y': 'Average Revenue'}
)

fig.update_layout(xaxis_tickangle=-45, title_x=0.5)
fig.show()
fig.write_image("genre_by_average_revenue.png")

##### Time Series Analysis

Movies Released Per Year

In [ ]:
year_count = data['release_year'].value_counts().sort_index()

fig = px.line(
    x = year_count.index,
    y = year_count.values,
    title='Number of Movies Released Per Year',
    labels={'x': 'Year', 'y': 'Number of Movies'}
)

fig.update_layout(title_x = 0.5)
fig.show()
fig.write_image("movie_released_per_year.png")

Insights from Movie Release per year : 

The film industry has expanded rapidly over the past decades, driven by digital production technologies and global distribution platforms.

Correlation and Statestical Analysis

In [ ]:
numeric_data = data.select_dtypes(include=np.number)

numeric_data = numeric_data.drop(columns=['id'], errors='ignore')

numeric_data.head()

# Pearson Correlation Matrix

corr_matrix = numeric_data.corr(method='pearson')
corr_matrix.head()

# Correlation Heatmap

plt.figure(figsize=(12,8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    linewidths=0.5
)

plt.title('Pearson Correlation Heatmap')
plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.show()

Correlation Insights : 

Budget ↔ Revenue → Strong positive

Vote Count ↔ Popularity → Moderate positive

Popularity ↔ Revenue → Moderate positive

Vote Average ↔ Revenue → Weak

Budget & Revenue Insights : 

Strong Positive Correlation - Budget shows a strong positive correlation with revenue, indicating that higher production investment often leads to greater box office returns.

Vote Average vs Revenue Insights : 

Weak Correlation - Critical or audience ratings do not strongly determine commercial success. Financial performance is influenced more by marketing, franchise value, and distribution scale.

Correlation of Budget and Revenue

In [ ]:
numeric_data = ['budget','revenue','vote_average', 'vote_count', 'popularity_scaled']

plt.figure(figsize=(12, 6))

sns.boxplot(data=data[numeric_data])

plt.title('Box Plot of Numeric Features')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# If i used this data then box plot gives right skewed diagram so i have to transform it using log transformation

data['budget_log'] = np.log1p(data['budget'])
data['revenue_log'] = np.log1p(data['revenue'])
data['vote_count_log'] = np.log1p(data['vote_count'])

log_cols = ['budget_log','revenue_log','vote_count_log',
            'vote_average','popularity_scaled']

plt.figure(figsize=(12, 6))

sns.boxplot(data=data[log_cols])

plt.title('Box Plot (Log Transformed Features)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Insights :

Heavy presence of outliers in revenue & budget so Log transformation improves interpretability of data so use Log Transformation.

The presence of extreme outliers confirms that blockbuster movies significantly distort overall distribution patterns.

Runtime vs Revenue :

In [ ]:
data['budget_log'] = np.log1p(data['budget'])
data['revenue_log'] = np.log1p(data['revenue'])

df_fin = data[(data['budget'] > 0) & (data['revenue'] > 0)]

df_fin = df_fin.explode('genres')

fig = px.scatter(
    df_fin,
    x='runtime',
    y='revenue',
    color='genres',
    title='Runtime vs Revenue'
)

fig.show()
fig.write_image("runtime_revenue_by_genre.png")

Country Collaboration Network

which countries collaborate most?

In [ ]:
countries = data.explode('production_countries')

country_counts = (
    countries['production_countries'].value_counts().head(20).reset_index()
)

country_counts.columns = ['Country', 'Movie_Count']

country_counts = country_counts.sort_values(by='Movie_Count')

plt.figure(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(country_counts)))

bars = plt.barh(
    country_counts['Country'],
    country_counts['Movie_Count'],
    color=colors
)

plt.title('Top Film Producing Countries')
plt.xlabel('Movie Counts')
plt.ylabel('Countries')

for i, value in enumerate(country_counts.Movie_Count):
    plt.text(value, i, f'{value}', va='center')

plt.show()

Production Company Genre Specialization : 

Which companies specialize in which genres?

In [ ]:
company_genre = (
    data.explode('production_companies').explode('genres').groupby(['production_companies', 'genres']).size().unstack(fill_value=0)
)

top_companies = (company_genre.sum(axis=1).sort_values(ascending=False).head(10).index)

company_genre_top = company_genre.loc[top_companies]

plt.figure(figsize=(15, 8))
sns.heatmap(company_genre_top, cmap='YlGnBu', annot=True, fmt='d')  
plt.title("Top 10 Companies vs Genre")
plt.show()

Genre wise Top Countries

In [ ]:
country_genre = (
    data
    .explode('production_countries')
    .explode('genres')
)

country_genre_count = (
    country_genre.groupby(['genres', 'production_countries']).size().reset_index(name='movie_count')
)

top3_country_per_genre = (
    country_genre_count.sort_values(['genres', 'movie_count'], ascending=[True, False]).groupby('genres').head(3)
)

fig = px.bar(
    top3_country_per_genre,
    x='movie_count',
    y='genres',
    color='production_countries',
    orientation='h',
    title="Top 3 Producing Countries per Genres"
)

fig.show()
fig.write_image("top_producing_countries.png")

Distribution of Movie Runtime

In [ ]:
data = data[data['runtime'] <= 300]

sns.boxplot(data['runtime'])
plt.title('Box plot for Movie Runtime')

runtime_data = data['runtime'].dropna()

plt.figure(figsize=(10,6))

plt.hist(runtime_data, bins=40, edgecolor='black')

plt.title("Distribution of Movie Runtime")
plt.xlabel("Runtime (minutes)")
plt.ylabel("Number of Movies")

plt.show()

Genre Distribution by Season (Month vs Genre)

In [ ]:
genres_exploded = data.explode('genres')

month_genre = (
    genres_exploded
    .groupby(['release_month','genres'])
    .size()
    .reset_index(name='count')
)

pivot = month_genre.pivot(
    index='genres',
    columns='release_month',
    values='count'
)

fig = px.imshow(
    pivot,
    color_continuous_scale='viridis',
    text_auto=True,
    aspect="auto",
    labels=dict(x="Month", y="Genre", color="Movie Count"),
    title="Genre Distribution By Month"
)

fig.update_layout(
    template = 'plotly_white'
)

fig.show()
fig.write_image("genre_dist_by_season.png")

Runtime Distribution By Genre

In [ ]:
genre_df = data.copy()

genre_df = genre_df.reset_index(drop=True)

genre_df = genre_df.explode('genres')

genre_df = genre_df.dropna(subset=['runtime', 'genres'])

genre_df['genres'] = genre_df['genres'].astype(str)
genre_df['runtime'] = pd.to_numeric(genre_df['runtime'], errors='coerce')

genre_df = genre_df.drop_duplicates(subset=['genres', 'runtime'])

fig = px.box(
    genre_df,
    x="genres",
    y="runtime",
    color="genres",
    title="Runtime Distribution by Genre"
)

fig.show()
fig.write_image("runtime_distribution_by_genre.png")

Franchise Detection Analysis

In [ ]:
import re

pattern = r'(?:\b\d+\b|II|III|IV|V|VI|Part|Chapter)'

sequels = data[data['title'].str.contains(pattern, case=False, na=False)].copy()

sequels['franchise'] = (
    sequels['title'].str.replace(pattern, '', regex=True).str.strip()
)

sequels = sequels[(sequels['franchise'] != '') & (sequels['franchise'] != ':') & (sequels['franchise'] != '/')]

franchise_count = (
    sequels['franchise'].value_counts().head(15).reset_index()
)

franchise_count.columns = ['franchise','movie_count']

fig = px.bar(
    franchise_count,
    x='movie_count',
    y='franchise',
    orientation='h',
    title='Top Movie Franchises (Approximation)',
    color='movie_count',
    color_continuous_scale='viridis'
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()
fig.write_image("franchise_detection.png")

Production Company Specialization

In [ ]:
company_genre = data.explode('production_companies').explode('genres')

company_genre_count = (
    company_genre.groupby(['production_companies', 'genres']).size().reset_index(name='movie_count')
)

top_companies = company_genre_count.groupby('production_companies')['movie_count'].sum().nlargest(10).index

filtered = company_genre_count[company_genre_count['production_companies'].isin(top_companies)]

pivot = filtered.pivot_table(
    index='production_companies', columns='genres', values='movie_count', fill_value=0
)

fig = px.imshow(
    pivot,
    color_continuous_scale='viridis',
    text_auto=True,
    aspect=True,
    labels=dict(x="Genres", y="Production Company", color="Movie Count"),
    title='Production Company Genre Specialization'
)

fig.show()
fig.write_image("production_company_specialization.png")

Keywords trend over time

In [ ]:
data['year'] = data['release_date'].dt.year

keywords_df = data.explode('keywords')

top_keywords = keywords_df['keywords'].value_counts().head(10).index

trend_data = keywords_df[keywords_df['keywords'].isin(top_keywords)]

keyword_trend = (trend_data.groupby(['year', 'keywords']).size().reset_index(name='movie_count'))

fig = px.line(
    keyword_trend,
    x='year',
    y='movie_count',
    color='keywords',
    title='Keyword Trend Over Time'
)

fig.show()
fig.write_image("keywords_trends_over_time.png")

Movie Production Growth by Country

In [ ]:
countries = data.explode('production_countries')

country_year = countries.groupby(['release_year', 'production_countries']).size().reset_index(name='movies')

fig = px.line(
    country_year,
    x='release_year',
    y='movies',
    color='production_countries',
    title='Movie Production by Country Over Time'
)

fig.show()
fig.write_image("movie_production_by_country.png")